data cleaning

In [1]:
import pandas as pd

df = pd.read_csv('iot_telemetry_data.csv')

# 1. Check the structure and data types
print("--- Data Info ---")
print(df.info())

# 2. Look at the actual data in the first 5 rows
print("\n--- First 5 Rows ---")
display(df.head())

--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 405184 entries, 0 to 405183
Data columns (total 9 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   ts        405184 non-null  float64
 1   device    405184 non-null  object 
 2   co        405184 non-null  float64
 3   humidity  405184 non-null  float64
 4   light     405184 non-null  bool   
 5   lpg       405184 non-null  float64
 6   motion    405184 non-null  bool   
 7   smoke     405184 non-null  float64
 8   temp      405184 non-null  float64
dtypes: bool(2), float64(6), object(1)
memory usage: 22.4+ MB
None

--- First 5 Rows ---


,ts,device,co,humidity,light,lpg,motion,smoke,temp
0,1.594512e+09,b8:27:eb:bf:9d:51,0.004956,51.000000,False,0.007651,False,0.020411,22.700000
1,1.594512e+09,00:0f:00:70:91:0a,0.002840,76.000000,False,0.005114,False,0.013275,19.700001
2,1.594512e+09,b8:27:eb:bf:9d:51,0.004976,50.900000,False,0.007673,False,0.020475,22.600000
3,1.594512e+09,1c:bf:ce:15:ec:4d,0.004403,76.800003,True,0.007023,False,0.018628,27.000000
4,1.594512e+09,b8:27:eb:bf:9d:51,0.004967,50.900000,False,0.007664,False,0.020448,22.600000


In [2]:
import pandas as pd


df = pd.read_csv('iot_telemetry_data.csv')

# 1. Fix the Timestamp column: Convert from Unix float to standard SQL datetime string
df['ts'] = pd.to_datetime(df['ts'], unit='s').dt.strftime('%Y-%m-%d %H:%M:%S')

# 2. Fix the Boolean columns: Convert Python's 'True'/'False' to lowercase 'true'/'false'
df['light'] = df['light'].astype(str).str.lower()
df['motion'] = df['motion'].astype(str).str.lower()

# 3. Save to a new CSV file (Critically, we set index=False so we don't accidentally add an extra row number column)
df.to_csv('iot_telemetry_data_clean.csv', index=False)

# Show the first 5 rows of the fixed data to verify
print("--- Cleaned Data ---")
display(df.head())

--- Cleaned Data ---


,ts,device,co,humidity,light,lpg,motion,smoke,temp
0,2020-07-12 00:01:34,b8:27:eb:bf:9d:51,0.004956,51.000000,false,0.007651,false,0.020411,22.700000
1,2020-07-12 00:01:34,00:0f:00:70:91:0a,0.002840,76.000000,false,0.005114,false,0.013275,19.700001
2,2020-07-12 00:01:38,b8:27:eb:bf:9d:51,0.004976,50.900000,false,0.007673,false,0.020475,22.600000
3,2020-07-12 00:01:39,1c:bf:ce:15:ec:4d,0.004403,76.800003,true,0.007023,false,0.018628,27.000000
4,2020-07-12 00:01:41,b8:27:eb:bf:9d:51,0.004967,50.900000,false,0.007664,false,0.020448,22.600000


# Question 5: PySpark Machine Learning

In [5]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. Initialize Spark Session
spark = SparkSession.builder.appName("IoTPredictiveAnalysis").getOrCreate()

# 2. Load Data
df = spark.read.csv("iot_telemetry_data_clean.csv", header=True, inferSchema=True)

# 3. Data Preparation
# Convert boolean 'light' column to a numeric label (0.0 or 1.0)
df = df.withColumn("light_str", df["light"].cast("string"))
indexer = StringIndexer(inputCol="light_str", outputCol="label")
df_indexed = indexer.fit(df).transform(df)

# Assemble features into a single vector column
assembler = VectorAssembler(inputCols=["temp", "humidity", "co", "lpg"], outputCol="features")
data = assembler.transform(df_indexed).select("features", "label")

# 4. Train/Test Split
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

# 5. Train Logistic Regression Model
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
lr_model = lr.fit(train_data)

# 6. Make Predictions
predictions = lr_model.transform(test_data)

# 7. Evaluate Model Accuracy
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
roc_auc = evaluator.evaluate(predictions)

print(f"Model ROC-AUC Score: {roc_auc:.4f}")

Model ROC-AUC Score: 0.9637
